# Chapter 4 — Spectroscopic binaries and radial-velocity variability

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/vasilybelokurov/gaia-nss-tutorial/blob/main/notebooks/04_spectroscopic_binaries.ipynb)

**Goal.** Gaia's Radial Velocity Spectrometer (RVS) measured, for stars brighter than $G_{\rm RVS} \approx 14$, the
radial velocity at each transit. For 186 905 stars DR3 published a spectroscopic orbit (Gosset et al. 2025,
https://ui.adsabs.harvard.edu/abs/2025A&A...693A.124G), and for every star with $G_{\rm RVS} \le 12$ two indicators of
whether its velocity varied (Katz et al. 2023, https://ui.adsabs.harvard.edu/abs/2023A&A...674A...5K). We use the
orbits to see a classic result of binary-star physics, tidal circularisation, and test the variability indicators
against an independent multi-epoch survey, APOGEE.

**You will learn**
1. what a single-lined spectroscopic orbit measures ($K_1$, $P$, $e$), and the spectroscopic mass function;
2. how the eccentricities of binaries depend on period and on the stellar envelope (tidal circularisation);
3. to select radial-velocity-variable stars from `gaia_source`, and where the indicators are valid;
4. to validate them against APOGEE DR17, and why a validation sample needs the same scrutiny as the sample itself.

In [ ]:
import hashlib, json, urllib.request
from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from matplotlib.colors import LogNorm

# --- Data access (identical in every chapter) -------------------------------------------------------------
DATA_URL = 'https://github.com/vasilybelokurov/gaia-nss-tutorial/releases/download/data-v0.2/'
_local = Path('../data/MANIFEST.json')            # present when you run inside a clone of the repository
MANIFEST = json.loads(_local.read_text() if _local.exists() else urllib.request.urlopen(DATA_URL + 'MANIFEST.json').read())


def load(name, cache=Path('data')):
    """Download <name>.parquet from the tutorial's data release (again if the local copy is stale), check its SHA-256."""
    path = cache / f'{name}.parquet'
    if not path.exists():                          # first use: fetch the file into ./data
        cache.mkdir(exist_ok=True)
        print(f'downloading {path.name} ...')
        urllib.request.urlretrieve(DATA_URL + path.name, path)
    if hashlib.sha256(path.read_bytes()).hexdigest() != MANIFEST[name]['sha256']:   # stale or damaged copy
        print(f'downloading {path.name} again (checksum mismatch) ...')
        urllib.request.urlretrieve(DATA_URL + path.name, path)
    # the checksum guarantees that everyone runs this chapter on exactly the same rows
    assert hashlib.sha256(path.read_bytes()).hexdigest() == MANIFEST[name]['sha256'], f'{path} does not match the manifest'
    return pd.read_parquet(path)


tbo = load('nss_two_body')
sb1 = tbo[tbo.nss_solution_type.isin(['SB1', 'SB1C'])].reset_index(drop=True)
print(f"{len(sb1):,} SB1 solutions, of which {(sb1.nss_solution_type == 'SB1C').sum()} circular (SB1C)")

## 4.1 What a spectroscopic orbit measures

The radial velocity of the primary of a binary varies as

$$ v_r(t) = \gamma + K_1\,[\cos(\nu + \omega) + e\cos\omega], $$

with $\gamma$ the systemic velocity, $\nu(t)$ the true anomaly, $\omega$ the argument of periastron and $K_1$ the
semi-amplitude. Only the line-of-sight component of the motion is seen, so $K_1 = 2\pi a_1\sin i / [P\sqrt{1-e^2}]$
contains the unknown inclination. Combined with Kepler's third law it gives the **spectroscopic mass function**

$$ f(M) = \frac{P K_1^3 (1-e^2)^{3/2}}{2\pi G} = \frac{(M_2 \sin i)^3}{(M_1 + M_2)^2}, \qquad
f(M)\,[M_\odot] = 1.0361\times 10^{-7}\,(1-e^2)^{3/2}\,(K_1/{\rm km\,s^{-1}})^3\,(P/{\rm d}) . $$

Since $\sin i \le 1$, $f(M)$ is a strict lower limit on $M_2^3/(M_1+M_2)^2$. Compare the astrometric mass function of
Chapter 3, which contains no inclination.

In [ ]:
sb1['fM'] = 1.0361e-7 * (1 - sb1.eccentricity.fillna(0)**2)**1.5 * sb1.semi_amplitude_primary**3 * sb1.period
fig, ax = plt.subplots(1, 2, figsize=(13, 4.5))
ax[0].hist2d(np.log10(sb1.period), np.log10(sb1.semi_amplitude_primary), bins=150, norm=LogNorm(), cmap='viridis')
ax[0].set(xlabel='log$_{10}$ P [d]', ylabel='log$_{10}$ $K_1$ [km/s]')
ax[1].hist(np.log10(sb1.fM), 150, histtype='step', lw=2)
ax[1].set(xlabel='log$_{10}$ f(M) [$M_\\odot$]', ylabel='SB1 solutions per bin')
plt.tight_layout(); plt.show()
# identity: Kepler's third law, checked on one solar-mass companion to the Sun seen edge-on at P = 1 yr:
# K1 = 2 pi a1 / P with a1 = 0.5 au (mass ratio 1) and M1 + M2 = 2 Msun -> f(M) = 1 / 4 Msun
a_rel = (2 * 1.0**2)**(1 / 3)                         # au
K = 2 * np.pi * (a_rel / 2) * 1.495978707e8 / (365.25 * 86400)   # km/s
assert abs(1.0361e-7 * K**3 * 365.25 - 0.25) < 1e-3

**Figure 4.1.** Left: semi-amplitude $K_1$ against period for the SB1 solutions (log-scaled counts). Right: distribution
of their spectroscopic mass functions.

The upper envelope of $K_1$ falls with period, as Kepler's law requires ($K_1 \propto P^{-1/3}$ at fixed masses); the
lower edge, a few km/s, is set by the precision of the RVS epoch velocities, and drops at the longest periods, where
more epochs cover the orbit. A narrow vertical gap at $\log P \approx 2.26$ (about 180 d, half a year) is a period where
few solutions survived validation, as in the astrometric orbits of Chapter 3. The mass functions peak near
$0.03\,M_\odot$: for a solar-mass primary that is $M_2 \sin i \approx 0.3\,M_\odot$, ordinary M-dwarf companions.

## 4.2 Tidal circularisation

Tides raised by each star on the other dissipate orbital energy and circularise the orbit, much faster in close
binaries. Stars with deep convective envelopes (cooler than ~6250 K on the main sequence) dissipate tides more
efficiently than hot stars with radiative envelopes. The expectation: below some cut-off period, orbits are circular.
Using the upper envelope of the Gaia period–eccentricity diagram, Bashi et al. (2023,
https://ui.adsabs.harvard.edu/abs/2023MNRAS.522.1184B) found that the cut-off decreases from 6.5 d at
$T_{\rm eff} \approx 5700$ K to 2.5 d at 6800 K, and does not depend on age. Figure 4.2 shows eccentricity against period
for SB1 solutions of main-sequence stars, split by the temperature of the RVS template.

A caution before reading the plot: eccentricity is positive by definition, so noise biases small eccentricities up
(Lucy & Sweeney 1971, https://ui.adsabs.harvard.edu/abs/1971AJ.....76..544L). With 20–40 epochs, a circular orbit is often
fitted with $e \sim 0.05$. Gaia published `SB1C` (circular) solutions only when the eccentric fit was not significantly
better; we plot them at $e = 0$.

In [ ]:
sb1['MG'] = sb1.phot_g_mean_mag + 5 * np.log10(sb1.parallax) - 10
dwarf = (sb1.parallax / sb1.parallax_error > 10) & (sb1.MG > 2.0) & (sb1.MG < 3.0 + 3.2 * (sb1.bp_rp - 0.4)) \
        & (sb1.MG > 3.2 * (sb1.bp_rp - 0.4) - 1.5)                      # a broad main-sequence band
e = sb1.eccentricity.fillna(0)                                         # SB1C: e = 0
cool = dwarf & (sb1.rv_template_teff < 6000)
hot = dwarf & (sb1.rv_template_teff >= 6500)
pb = np.logspace(0, 3, 13)
fig, ax = plt.subplots(1, 2, figsize=(14, 4.8), sharey=True)
med = {}
for a, (lab, s) in zip(ax, [('cool dwarfs, $T_{\\rm eff}$ < 6000 K', cool), ('hot dwarfs, $T_{\\rm eff}$ ≥ 6500 K', hot)]):
    a.plot(sb1.period[s], e[s], ',', alpha=0.4, color='k')
    m = [np.median(e[s & (sb1.period >= lo) & (sb1.period < hi)]) for lo, hi in zip(pb[:-1], pb[1:])]
    med[lab] = np.array(m)
    a.plot(np.sqrt(pb[:-1] * pb[1:]), m, 'ro-', label='median in period bins')
    a.set(xscale='log', xlim=(1, 1000), ylim=(-0.02, 1), xlabel='period [d]', title=f'{lab}: {s.sum():,} SB1')
ax[0].set_ylabel('eccentricity'); ax[0].legend()
plt.tight_layout(); plt.show()
mc, mh = list(med.values())
print('median e, cool:', np.round(mc, 3), '\nmedian e, hot: ', np.round(mh, 3))
pc = np.sqrt(pb[:-1] * pb[1:])                                        # bin centres [d]
# descriptive (frozen after a development run): in both groups the median eccentricity is below 0.1 at 3-8 d and above
# 0.2 at 30-1000 d
for m in (mc, mh):
    assert np.all(m[(pc > 3) & (pc < 8)] < 0.1) and np.all(m[pc > 30] > 0.2)

**Figure 4.2.** Eccentricity against period for SB1 solutions (SB1C at $e = 0$) of main-sequence stars with parallax
S/N > 10, for RVS template temperatures below 6000 K (left) and at or above 6500 K (right). Red: median eccentricity in
12 logarithmic period bins.

In both groups the median eccentricity is below 0.1 for periods of about 3–8 d and rises to 0.25–0.28 above 30 d:
close binaries are on nearly circular orbits, as tidal circularisation predicts. Two features need care. The scatter
of points at $e \approx 0.05$ below 10 d, rather than at $e = 0$, is the Lucy–Sweeney bias. And for cool dwarfs the
median rises again below 3 d, to 0.18 at 1–2 d, which tidal theory does not predict; we have not investigated these
solutions, and they should be checked before any conclusion is drawn from them. Medians are
also the wrong statistic for a cut-off: Bashi et al. measured the *upper envelope* of eccentricity after quality cuts,
and found the temperature dependence that this plot does not resolve (Exercise 2).

## 4.3 Radial-velocity variability without an orbit

Most binaries have no Gaia orbit: too few epochs, too long a period, or too small an amplitude. But for every star with
$G_{\rm RVS} \le 12$ that was measured on the RVS's single-star pipeline (`rv_method_used` = 1), DR3 gives:

- `rv_chisq_pvalue`: the probability of the observed scatter of the epoch velocities if the star had a constant
  velocity;
- `rv_renormalised_gof`: a normalised goodness of fit of the constant-velocity model, ~N(0, 1) for constant stars;
- `rv_amplitude_robust`: the range of the epoch velocities after removing the most extreme one (not $2K_1$).

Katz et al. (2023, Sect. 3.7) recommend a conservative criterion for variable stars: `rv_nb_transits` ≥ 10,
3900 ≤ `rv_template_teff` ≤ 8000 K, `rv_chisq_pvalue` ≤ 0.01 and `rv_renormalised_gof` > 4. Outside this temperature range
the epoch uncertainties are less reliable.

To test the criterion we need an independent measurement of velocity variability. APOGEE DR17 (Abdurro'uf et al. 2022,
https://ui.adsabs.harvard.edu/abs/2022ApJS..259...35A) observed many stars several times at a resolution of 22 500
and reports `vscatter`, the standard deviation of the visit velocities. The file `apogee_gaia` holds APOGEE stars with
at least three visits and S/N > 50, joined with their Gaia RV indicators.

In [ ]:
ap = load('apogee_gaia')
g = ap[(ap.rv_method_used == 1) & (ap.rv_nb_transits >= 10)].copy()
g['variable'] = (g.rv_chisq_pvalue <= 0.01) & (g.rv_renormalised_gof > 4) & g.rv_template_teff.between(3900, 8000)
vb = [0, 0.3, 1, 3, 10, 30, 1000]                                     # bins of APOGEE vscatter [km/s]
g['vbin'] = pd.cut(g.vscatter, vb)
groups = {'cool dwarfs (APOGEE $T_{\\rm eff}$ < 6000 K, log g > 3.5)': (g.teff < 6000) & (g.logg > 3.5),
          'giants ($T_{\\rm eff}$ < 5200 K, log g < 3.5)': (g.teff < 5200) & (g.logg < 3.5),
          'hot stars ($T_{\\rm eff}$ > 6500 K)': g.teff > 6500}
x = np.sqrt(np.array(vb[:-1]).clip(0.1) * np.array(vb[1:]))
frac = {}
for lab, s in groups.items():
    t = g[s].groupby('vbin', observed=False)['variable'].agg(['sum', 'size'])
    p = t['sum'] / t['size']
    err = np.sqrt(p * (1 - p) / t['size'])                            # binomial standard error
    frac[lab] = (p.to_numpy(), err.to_numpy(), t['size'].to_numpy())
    plt.errorbar(x, p, err, fmt='o-', label=f'{lab}: {s.sum():,} stars')
plt.xscale('log'); plt.xlabel('APOGEE vscatter [km/s]'); plt.ylabel('fraction RV-variable in Gaia (Katz et al. criterion)')
plt.ylim(0, 1.02); plt.legend(fontsize=8); plt.show()
for lab, (p, err, n) in frac.items():
    print(f'{lab}: {np.round(p, 3)}  (N = {n})')
# descriptive: for cool dwarfs and giants, the Gaia variable fraction rises with APOGEE vscatter from bin to bin, up to
# 30 km/s, by more than 3 binomial standard errors (the last bin, near 100%, holds few stars)
for lab in list(groups)[:2]:
    p, err, n = frac[lab][0][:5], frac[lab][1][:5], frac[lab][2][:5]
    assert np.all(np.diff(p) > 3 * np.hypot(err[1:], err[:-1])), lab

**Figure 4.3.** Fraction of APOGEE stars flagged as RV-variable by the Gaia DR3 criterion of Katz et al. (2023), in
bins of APOGEE `vscatter` (0–0.3, 0.3–1, 1–3, 3–10, 10–30, > 30 km/s), for cool dwarfs, giants and hot stars (APOGEE
temperatures and gravities). Error bars: binomial standard errors. Only stars with `rv_method_used` = 1 and at least
10 RVS transits.

For cool dwarfs and giants, the Gaia criterion agrees with APOGEE: 4–6% of the stars APOGEE finds constant
(vscatter < 0.3 km/s) are flagged, rising to 36–46% at 0.3–1 km/s, 65% at 1–3 km/s, 85–89% at 3–10 km/s and 92–98% above
10 km/s. The rise reflects the RVS precision: velocity changes below ~1 km/s are hard to detect with 10–40 epochs at
$G_{\rm RVS} \le 12$. Part of the 4–6% flagged among "constant" APOGEE stars will be real binaries that APOGEE's few
visits (median 3) missed.

The hot stars behave differently: APOGEE reports large vscatter for most of them (median v sin i of the stars in the
3–10 km/s bin is ~60 km/s), but Gaia flags only 6–10%. Broad lines of fast rotators make single-visit velocities
uncertain, and APOGEE's `vscatter` does not account for that, while the Katz et al. criterion compares the scatter
with each epoch's uncertainty and excludes stars hotter than 8000 K. Here it is the validation sample, not Gaia, that
misleads: an independent catalogue has its own failure modes, and these must be understood before it is used as truth.

## Summary

- A spectroscopic orbit gives $K_1$, $P$, $e$; its mass function is a lower limit because of the unknown inclination.
- SB1 eccentricities fall towards short periods (median below 0.1 at 3–8 d, 0.25–0.28 above 30 d): close binaries are
  circular. Measuring the cut-off period needs the upper envelope and clean solutions.
- For $G_{\rm RVS} \le 12$, `rv_chisq_pvalue` and `rv_renormalised_gof` flag RV-variable stars. The Katz et al.
  criterion agrees with APOGEE for cool stars; validation samples have their own failure modes.

**Exercises.** (1) Using `rv_amplitude_robust`, estimate the minimum companion mass of the Gaia-variable cool dwarfs,
assuming $K_1 \ge$ amplitude/2 and $P$ = the RVS time span (`rv_time_duration`). Which assumption is the weak one?
(2) Measure the upper envelope (e.g. the 90th percentile of $e$) in narrow period bins for 5500–5900 K and 6500–7000 K,
after keeping only solutions with `significance` > 20. Do you recover the trend of Bashi et al. (2023)? (3) How many of the APOGEE stars with
vscatter > 10 km/s have an SB1 orbit in Gaia (`non_single_star`)?

## References

- Gosset et al. (2025), *Gaia Data Release 3: Spectroscopic binary-star orbital solutions: The SB1 processing chain*, A&A 693, A124: https://ui.adsabs.harvard.edu/abs/2025A&A...693A.124G
- Gaia Collaboration, Arenou et al. (2023), *Gaia Data Release 3. Stellar multiplicity, a teaser for the hidden treasure*, A&A 674, A34: https://ui.adsabs.harvard.edu/abs/2023A&A...674A..34G
- Katz et al. (2023), *Gaia Data Release 3. Properties and validation of the radial velocities*, A&A 674, A5: https://ui.adsabs.harvard.edu/abs/2023A&A...674A...5K
- Lucy & Sweeney (1971), *Spectroscopic binaries with circular orbits*, AJ 76, 544: https://ui.adsabs.harvard.edu/abs/1971AJ.....76..544L
- Bashi et al. (2022), *Gaia spectroscopic orbits validated with LAMOST and GALAH radial velocities*, MNRAS 517, 3888: https://ui.adsabs.harvard.edu/abs/2022MNRAS.517.3888B
- Bashi et al. (2023), *Features of Gaia DR3 spectroscopic binaries I. Tidal circularization of main-sequence stars*, MNRAS 522, 1184: https://ui.adsabs.harvard.edu/abs/2023MNRAS.522.1184B
- Abdurro'uf et al. (2022), *The Seventeenth Data Release of the Sloan Digital Sky Surveys (APOGEE-2 DR17)*, ApJS 259, 35: https://ui.adsabs.harvard.edu/abs/2022ApJS..259...35A